In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from statsmodels.tsa.statespace.sarimax import SARIMAX
from datetime import datetime
import sys

# Get forecast steps from command line
forecast_steps = int(sys.argv[1])

# Load and prepare historical data
data = pd.read_csv('blood_inventory_handover.csv', parse_dates=['Month'], index_col='Month')
data = data.asfreq('MS')

# Fit SARIMA model
p, d, q = 1, 0, 0
P, D, Q, S = 0, 1, 0, 12
model = SARIMAX(data['Total Given Units'], order=(p, d, q), seasonal_order=(P, D, Q, S))
results = model.fit()

# Forecast start date
current_date = pd.Timestamp(datetime.now().strftime('%Y-%m-01'))
forecast_start_date = data.index[-1] + pd.DateOffset(months=1)
start_date = max(current_date, forecast_start_date)

# Generate forecast dates and forecasts
forecast_dates = pd.date_range(start=start_date, periods=forecast_steps, freq='MS')
forecast = results.get_forecast(steps=forecast_steps)
forecast_df = forecast.summary_frame()
forecast_df.index = forecast_dates

# Save forecast to file
forecast_df['Date'] = forecast_dates
forecast_df.set_index('Date', inplace=True)
forecast_df.to_csv('sample_forecast_file_totalgivenunits.csv')

# Combine observed and forecasted data for plotting
combined_data = pd.concat([data['Total Given Units'], forecast_df['mean']])

# Plotting
plt.figure(figsize=(12, 6))
plt.plot(combined_data, label='Observed + Forecast', color='blue', linestyle='-', marker='o')
plt.plot(data['Total Given Units'], label='Observed', color='blue', linestyle='-', marker='o')
plt.plot(forecast_df['mean'], label='Forecast', color='orange', linestyle='--', marker='o')

# Confidence interval
plt.fill_between(forecast_df.index, forecast_df['mean_ci_lower'], forecast_df['mean_ci_upper'], 
                 color='orange', alpha=0.3, label='Forecast Range (Confidence Interval)')

# X-axis format
plt.gca().xaxis.set_major_locator(mdates.MonthLocator())
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=45)

# Labels, grid, legend
plt.title('Monthly Observed and Forecasted Blood Handover')
plt.xlabel('Month')
plt.ylabel('Blood Handover')
plt.grid(visible=True, which='major', linestyle='--', linewidth=0.5)
plt.legend()
plt.tight_layout()
plt.savefig('graphs/blood_handover_forecast.png')
plt.close()

ValueError: invalid literal for int() with base 10: '--f=c:\\Users\\Acer\\AppData\\Roaming\\jupyter\\runtime\\kernel-v3c92d54d451fb52ffe4dc59b89cee668e8470a84f.json'